# 03 - Transformer with Defense 1 (Regularization)

Defense: L2 + Dropout + EarlyStopping

In [1]:
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
BASELINE_TEST_ACC = 0.9331
BASELINE_ATTACK = pd.DataFrame([
    {'attack':'threshold_loss','auc':0.4821,'accuracy':0.3099,'f1':0.3636},
    {'attack':'logistic','auc':0.4818,'accuracy':0.8028,'f1':0.0},
    {'attack':'shadow_meta','auc':0.6151,'accuracy':0.6485,'f1':0.6802},
])

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def build_transformer_reg(n_features, dropout=0.35, l2v=1e-4):
    reg = regularizers.l2(l2v)
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64, kernel_regularizer=reg)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu', kernel_regularizer=reg)(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(64, kernel_regularizer=reg)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def mia_features(p, y):
    p = np.clip(np.asarray(p), 1e-8, 1-1e-8)
    y = np.asarray(y)
    loss = -(y*np.log(p) + (1-y)*np.log(1-p))
    conf = np.maximum(p, 1-p)
    ent = -(p*np.log(p) + (1-p)*np.log(1-p))
    cor = ((p >= 0.5).astype(int) == y).astype(float)
    margin = np.abs(p - 0.5)
    return np.column_stack([loss, conf, ent, cor, margin])

def attack_row(name, y_true, y_pred, y_score):
    return {'attack':name,'auc':roc_auc_score(y_true,y_score),'accuracy':accuracy_score(y_true,y_pred),
            'precision':precision_score(y_true,y_pred,zero_division=0),'recall':recall_score(y_true,y_pred,zero_division=0),
            'f1':f1_score(y_true,y_pred,zero_division=0)}

def run_attacks(model, Xtr_seq, ytr, Xte_seq, yte, Xall, yall, seed):
    p_tr = model.predict(Xtr_seq, verbose=0).ravel()
    p_te = model.predict(Xte_seq, verbose=0).ravel()
    Fm, Fn = mia_features(p_tr, ytr), mia_features(p_te, yte)
    Xa = np.vstack([Fm, Fn]); ya = np.concatenate([np.ones(len(Fm),dtype=int), np.zeros(len(Fn),dtype=int)])
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.4, random_state=seed, stratify=ya)

    score_tr = -Xa_tr[:,0]; score_te = -Xa_te[:,0]
    thrs = np.linspace(score_tr.min(), score_tr.max(), 300)
    best_t, best_b = thrs[0], -1
    for t in thrs:
        pred = (score_tr >= t).astype(int)
        tpr = ((pred==1)&(ya_tr==1)).sum()/max((ya_tr==1).sum(),1)
        tnr = ((pred==0)&(ya_tr==0)).sum()/max((ya_tr==0).sum(),1)
        b = 0.5*(tpr+tnr)
        if b > best_b: best_b, best_t = b, t
    pred_thr = (score_te >= best_t).astype(int)
    r_thr = attack_row('threshold_loss', ya_te, pred_thr, score_te)

    lr = LogisticRegression(max_iter=1000, random_state=seed)
    lr.fit(Xa_tr, ya_tr)
    proba_lr = lr.predict_proba(Xa_te)[:,1]
    pred_lr = (proba_lr >= 0.5).astype(int)
    r_lr = attack_row('logistic', ya_te, pred_lr, proba_lr)

    Xab, Xaux, yab, yaux = train_test_split(Xall, yall, test_size=0.30, random_state=seed, stratify=yall)
    Xm, Xn, ym, yn = train_test_split(Xab, yab, test_size=0.45, random_state=seed+1, stratify=yab)
    sct = StandardScaler(); Xm_sc = sct.fit_transform(Xm).astype(np.float32); Xn_sc = sct.transform(Xn).astype(np.float32)
    Xm_seq = Xm_sc.reshape(-1,1,Xm_sc.shape[1]); Xn_seq = Xn_sc.reshape(-1,1,Xn_sc.shape[1])
    set_seed(seed+1000); tf.keras.backend.clear_session()
    tgt = build_transformer_reg(Xm_sc.shape[1], dropout=0.35, l2v=1e-4)
    tgt.fit(Xm_seq, ym, epochs=80, batch_size=16, verbose=0)
    ptm = tgt.predict(Xm_seq, verbose=0).ravel(); ptn = tgt.predict(Xn_seq, verbose=0).ravel()
    Xt = np.vstack([mia_features(ptm, ym), mia_features(ptn, yn)])
    yt = np.concatenate([np.ones(len(ym),dtype=int), np.zeros(len(yn),dtype=int)])

    shadowX, shadowY = [], []
    for i in range(10):
        xsm, xsn, ysm, ysn = train_test_split(Xaux, yaux, test_size=0.5, random_state=seed+100+i, stratify=yaux)
        scs = StandardScaler(); xsm_sc = scs.fit_transform(xsm).astype(np.float32); xsn_sc = scs.transform(xsn).astype(np.float32)
        xsm_seq = xsm_sc.reshape(-1,1,xsm_sc.shape[1]); xsn_seq = xsn_sc.reshape(-1,1,xsn_sc.shape[1])
        set_seed(seed+2000+i); tf.keras.backend.clear_session()
        sh = build_transformer_reg(xsm_sc.shape[1], dropout=0.35, l2v=1e-4)
        sh.fit(xsm_seq, ysm, epochs=40, batch_size=16, verbose=0)
        psm = sh.predict(xsm_seq, verbose=0).ravel(); psn = sh.predict(xsn_seq, verbose=0).ravel()
        shadowX.append(np.vstack([mia_features(psm, ysm), mia_features(psn, ysn)]))
        shadowY.append(np.concatenate([np.ones(len(ysm),dtype=int), np.zeros(len(ysn),dtype=int)]))

    Xs = np.vstack(shadowX); ys = np.concatenate(shadowY)
    meta = GradientBoostingClassifier(n_estimators=250, learning_rate=0.05, max_depth=3, random_state=seed)
    meta.fit(Xs, ys)
    sc = meta.predict_proba(Xt)[:,1]; pr = (sc >= 0.5).astype(int)
    r_sh = attack_row('shadow_meta', yt, pr, sc)

    return pd.DataFrame([r_thr, r_lr, r_sh]).sort_values('auc', ascending=False)

In [3]:
print('=== Load prepared data ===')
b = np.load('../data_preparation/prepared_oasis2_transformer.npz', allow_pickle=True)
X, y = b['X'], b['y']
X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']
X_train_seq, X_test_seq = b['X_train_seq'], b['X_test_seq']
SEED = int(b['seed'][0])
print(f'X={X.shape}, train={X_train.shape}, test={X_test.shape}')

=== Load prepared data ===
X=(354, 9), train=(70, 9), test=(284, 9)


In [4]:
print('=== Train Defense 1 model ===')
set_seed(SEED + 10); tf.keras.backend.clear_session()
model_def1 = build_transformer_reg(X_train.shape[1], dropout=0.35, l2v=1e-4)
es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
h = model_def1.fit(X_train_seq, y_train, epochs=140, batch_size=32, validation_split=0.2, callbacks=[es], verbose=0)
p_te = model_def1.predict(X_test_seq, verbose=0).ravel()
test_acc_def1 = accuracy_score(y_test, (p_te >= 0.5).astype(int))
print(f'test_acc_def1={test_acc_def1:.4f}, epochs={len(h.history["loss"])}')

=== Train Defense 1 model ===
test_acc_def1=0.9366, epochs=19


In [5]:
print('=== Run MIA attacks (Defense 1) ===')
attack_def1 = run_attacks(model_def1, X_train_seq, y_train, X_test_seq, y_test, X, y, SEED)
display(attack_def1.round(4))

=== Run MIA attacks (Defense 1) ===


,attack,auc,accuracy,precision,recall,f1
0,threshold_loss,0.5329,0.6197,0.1905,0.2857,0.2286
1,logistic,0.5144,0.8028,0.0000,0.0000,0.0000
2,shadow_meta,0.5134,0.5263,0.5489,0.7481,0.6332


In [6]:
print('=== Baseline vs Defense 1 ===')
cmp = BASELINE_ATTACK.merge(attack_def1[['attack','auc','accuracy','f1']], on='attack', suffixes=('_baseline','_defense1'))
cmp['delta_auc'] = cmp['auc_defense1'] - cmp['auc_baseline']
cmp['delta_accuracy'] = cmp['accuracy_defense1'] - cmp['accuracy_baseline']
cmp['delta_f1'] = cmp['f1_defense1'] - cmp['f1_baseline']
display(cmp.sort_values('attack').round(4))

mcmp = pd.DataFrame([{'model':'Transformer','test_acc_baseline':BASELINE_TEST_ACC,'test_acc_defense1':test_acc_def1,'delta_test_acc':test_acc_def1-BASELINE_TEST_ACC}])
display(mcmp.round(4))

=== Baseline vs Defense 1 ===


,attack,auc_baseline,accuracy_baseline,f1_baseline,auc_defense1,accuracy_defense1,f1_defense1,delta_auc,delta_accuracy,delta_f1
1,logistic,0.4818,0.8028,0.0000,0.5144,0.8028,0.0000,0.0326,0.0000,0.000
2,shadow_meta,0.6151,0.6485,0.6802,0.5134,0.5263,0.6332,-0.1017,-0.1222,-0.047
0,threshold_loss,0.4821,0.3099,0.3636,0.5329,0.6197,0.2286,0.0508,0.3098,-0.135


,model,test_acc_baseline,test_acc_defense1,delta_test_acc
0,Transformer,0.9331,0.9366,0.0035


In [7]:
print('=== Quick AUC summary ===')
quick_auc = cmp[['attack','auc_baseline','auc_defense1','delta_auc']].sort_values('delta_auc')
display(quick_auc.round(4))

=== Quick AUC summary ===


,attack,auc_baseline,auc_defense1,delta_auc
2,shadow_meta,0.6151,0.5134,-0.1017
1,logistic,0.4818,0.5144,0.0326
0,threshold_loss,0.4821,0.5329,0.0508
